# Notebook 05 — Classes & OOP patterns

| | |
|---|---|
| **Track** | AI Engineering · `03-python-foundations/exercises/` |
| **Level** | Intermediate |
| **Pattern** | *Concept* → *Runnable demo* → *Your turn* → *Solution* |

**Prerequisites:** `04-modules-json-pathlib.ipynb`

**Next up:** `06-exceptions-context-managers.ipynb`

---

## Learning objectives

- Model services as classes with explicit state.
- Know when `@staticmethod` / `@classmethod` helps.
- Prefer composition for small components (retriever, reranker).

## Table of contents

1. **Minimal service class**
2. **Composition**
3. **Progressive drills — counters → static helpers → reranked pipeline**
4. **Exercise — `TextSplitter`**

---

## How to use this notebook

1. Run cells **top to bottom** the first time through.
2. Re-run and **change values** to test your understanding.
3. Do **Your turn** cells before opening solutions (HTML `<details>` blocks or later cells).

---


## 1 · Service class

*Explanation:* Bundle config + methods — e.g., embedder client, chunker.


In [ ]:
class FakeEmbedder:
    def __init__(self, dim: int = 3) -> None:
        self.dim = dim

    def embed(self, texts: list[str]) -> list[list[float]]:
        return [[float(len(t) % self.dim)] * self.dim for t in texts]

print(FakeEmbedder(2).embed(["ab", "abc"]))


## 2 · Composition

*Explanation:* Small classes working together beat deep inheritance trees.


In [ ]:
class Retriever:
    def __init__(self, docs: list[str]) -> None:
        self.docs = docs

    def search(self, q: str, k: int = 2) -> list[str]:
        return [d for d in self.docs if q.lower() in d.lower()][:k]

class Pipeline:
    def __init__(self, retriever: Retriever) -> None:
        self.r = retriever

    def answer(self, q: str) -> str:
        return " | ".join(self.r.search(q))

print(Pipeline(Retriever(["RAG basics", "Vector DB"])).answer("rag"))


---

## Progressive drills — **easy → harder**

Classes bundle **telemetry**, **pure helpers**, and **pluggable stages** exactly like modular AI microservices.


### A · Easiest — rolling keyword counts (`Bag`-lite)

Track token frequencies before throwing tensors at anything.


In [ ]:
class KeywordCounter:
    def __init__(self) -> None:
        self.freq: dict[str, int] = {}

    def observe(self, token: str) -> None:
        self.freq[token] = self.freq.get(token, 0) + 1


counter = KeywordCounter()
for tok in ["embedding", "chunk", "embedding"]:
    counter.observe(tok)
print(counter.freq)


### B · Medium — `@staticmethod` pure normalizers

Keep unicode cleanup deterministic without constructing helper objects.


In [ ]:
class TextCleanup:
    @staticmethod
    def slug(label: str) -> str:
        return "-".join(label.lower().split())


print(TextCleanup.slug("Hybrid Vector Store"))


### C · Harder — inject a reranker strategy

Composition grows when retrieval feeds ranking modules.


In [ ]:
class ReverseRanker:
    def reorder(self, hits: list[str]) -> list[str]:
        return list(reversed(hits))


class ServedPipeline:
    def __init__(self, retriever: Retriever, ranker: ReverseRanker | None = None) -> None:
        self.retriever = retriever
        self.ranker = ranker or ReverseRanker()

    def answer(self, q: str) -> str:
        hits = self.retriever.search(q, k=5)
        hits = self.ranker.reorder(hits)
        return " | ".join(hits)


docs = Retriever(["alpha rag doc", "beta sql doc", "gamma rag advanced"])
print(ServedPipeline(docs).answer("rag"))


### Exercise — `TextSplitter`

Class `TextSplitter(max_chars)` with method `split(text: str) -> list[str]` splitting contiguous slices ≤ `max_chars`.


In [ ]:
class TextSplitter:
    def __init__(self, max_chars: int) -> None:
        self.max_chars = max_chars

    def split(self, text: str) -> list[str]:
        raise NotImplementedError


assert TextSplitter(4).split("abcdefghi") == ["abcd", "efgh", "i"]
print("OK")


### Solution — TextSplitter

<details>
<summary>Click to expand</summary>

```python
class TextSplitter:
    def __init__(self, max_chars: int) -> None:
        self.max_chars = max_chars

    def split(self, text: str) -> list[str]:
        step = self.max_chars
        return [text[i : i + step] for i in range(0, len(text), step)]
```

</details>
